<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/02%20bigquery/04_ENARES_2024_STAGE2_cloud_storage_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 04_ENARES_2024_STAGE2_cloud_storage_report.ipynb
# Stage 2 - Cloud Storage Closure Report
# ============================================================

from google.colab import auth, drive
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import os

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = input("Enter your Google Cloud PROJECT_ID: ").strip()

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"

os.makedirs(LOG_DIR, exist_ok=True)

print("Using project:", PROJECT_ID)
print("Using LOG_DIR:", LOG_DIR)

In [ ]:
# ============================================================
# 1. Load required Stage 2 outputs
# ============================================================

required_files = {
    "dataset_registry": "ENARES_2024_STAGE2_bigquery_dataset_registry.csv",
    "table_mapping": "ENARES_2024_STAGE2_crs04_bigquery_table_mapping.csv",
    "source_file_check": "ENARES_2024_STAGE2_source_file_check.csv",
    "raw_table_inventory": "ENARES_2024_STAGE2_raw_table_inventory.csv",
    "rowcount_validation": "ENARES_2024_STAGE2_rowcount_validation.csv",
    "schema_validation": "ENARES_2024_STAGE2_schema_validation.csv",
    "metadata_inventory": "ENARES_2024_STAGE2_metadata_inventory.csv",
}

loaded = {}
missing_files = []

for key, filename in required_files.items():
    path = f"{LOG_DIR}/{filename}"

    if os.path.exists(path):
        loaded[key] = pd.read_csv(path)
        print(f"Loaded: {filename}")
    else:
        missing_files.append(filename)
        print(f"MISSING: {filename}")

if missing_files:
    raise FileNotFoundError(
        "Missing required Stage 2 output files: " + ", ".join(missing_files)
    )

In [ ]:
# ============================================================
# 2. Validate Stage 2 pass/fail criteria
# ============================================================

rowcount_validation = loaded["rowcount_validation"]
schema_validation = loaded["schema_validation"]
metadata_inventory = loaded["metadata_inventory"]
dataset_registry = loaded["dataset_registry"]
source_file_check = loaded["source_file_check"]
raw_table_inventory = loaded["raw_table_inventory"]
table_mapping = loaded["table_mapping"]

rowcount_pass = bool(rowcount_validation["rowcount_match"].all())
schema_pass = bool(schema_validation["column_count_match"].all())

required_metadata_tables = {
    "metadata_crs04_variables",
    "metadata_crs04_value_labels",
    "metadata_crs04_missing_codes",
    "metadata_crs04_source_files",
}

metadata_pass = required_metadata_tables.issubset(
    set(metadata_inventory["table_name"])
)

datasets_pass = bool(
    (dataset_registry["status"] == "created_or_exists").all()
)

source_files_pass = bool(
    source_file_check["file_exists"].all()
)

stage2_pass = rowcount_pass and schema_pass and metadata_pass and datasets_pass and source_files_pass

print("rowcount_pass:", rowcount_pass)
print("schema_pass:", schema_pass)
print("metadata_pass:", metadata_pass)
print("datasets_pass:", datasets_pass)
print("source_files_pass:", source_files_pass)
print("stage2_pass:", stage2_pass)

In [ ]:
# ============================================================
# 3. Build report sections
# ============================================================

def df_to_markdown(df):
    return df.to_markdown(index=False)

report_date = datetime.now(timezone.utc).isoformat()

raw_tables_summary = raw_table_inventory[
    [
        "source_file",
        "chapter",
        "target_table",
        "sav_rows",
        "sav_columns",
        "bq_rows",
        "bq_columns",
    ]
].copy()

metadata_summary = metadata_inventory[
    ["table_name", "row_count", "column_count"]
].copy()

dataset_summary = dataset_registry[
    ["dataset_id", "location", "status"]
].copy()

source_summary = source_file_check[
    ["chapter", "source_file", "target_table", "file_exists", "file_size_bytes"]
].copy()

In [ ]:
# ============================================================
# 4. Generate auditable Stage 2 closure report
# ============================================================

report = f"""# ENARES 2024 CRS04 - Stage 2 Cloud Storage Report

Fecha de ejecucion: {report_date}

Google Cloud project_id: `{PROJECT_ID}`

Cuenta operativa esperada: `anacordero.001@gmail.com`

## Decision

Stage 2 pass: `{stage2_pass}`

## Validaciones automaticas

| Validacion | Resultado |
|---|---:|
| Rowcount .sav vs BigQuery | `{rowcount_pass}` |
| Schema / column count validation | `{schema_pass}` |
| Metadata preservation | `{metadata_pass}` |
| BigQuery datasets created or verified | `{datasets_pass}` |
| Source files found in Drive | `{source_files_pass}` |

## Datasets BigQuery

{df_to_markdown(dataset_summary)}

## Archivos fuente CRS04

{df_to_markdown(source_summary)}

## Tablas raw cargadas

{df_to_markdown(raw_tables_summary)}

## Metadata preservada

{df_to_markdown(metadata_summary)}

## Outputs obligatorios usados

- `ENARES_2024_STAGE2_bigquery_dataset_registry.csv`
- `ENARES_2024_STAGE2_crs04_bigquery_table_mapping.csv`
- `ENARES_2024_STAGE2_source_file_check.csv`
- `ENARES_2024_STAGE2_raw_table_inventory.csv`
- `ENARES_2024_STAGE2_rowcount_validation.csv`
- `ENARES_2024_STAGE2_schema_validation.csv`
- `ENARES_2024_STAGE2_metadata_inventory.csv`

## Limite metodologico Stage 2

En Stage 2 no se hizo merge, no se recodificaron variables, no se crearon outcomes, no se construyeron variables derivadas, no se calcularon indicadores y no se interpreto ningun resultado estadistico.

La capa producida es raw, trazable, completa y auditable en BigQuery.

## Pendientes para Fase 03

- Validar llave de merge: `ID`, `COLEGIAL_ID`, `ID + COLEGIAL_ID` u otra llave documentada en metadata.
- Revisar duplicados, no matches y cardinalidad antes de crear `cleaned_crs04_merged_adolescents`.
- Si `ID`, `COLEGIAL_ID` u otra llave llega como `FLOAT64`, verificar que no existan decimales reales antes de convertir a `INT64`.
- Traducir y preservar el diseno muestral complejo: `CCDD` como estrato, `ID`/colegio como UPM, `ID_AULA`/seccion como USM y peso muestral oficial.
- Verificar variables con prefijo `C3` dentro de CRS04 como items comunes CRS03/CRS04 antes de usarlas en variables derivadas.

## Decision final

Decision Stage 2: `{"Aprobado" if stage2_pass else "Pendiente de correcciones"}`

"""

report_output = f"{LOG_DIR}/ENARES_2024_STAGE2_cloud_storage_report.md"

with open(report_output, "w", encoding="utf-8") as f:
    f.write(report)

print(f"Report saved to: {report_output}")
print(report)

In [ ]:
# ============================================================
# 5. Final acceptance check
# ============================================================

if not os.path.exists(report_output):
    raise FileNotFoundError("Cloud storage report was not created.")

if not stage2_pass:
    print("Stage 2 report was created, but decision is PENDIENTE DE CORRECCIONES.")
    print("Review failed validations above before starting Fase 03.")
else:
    print("Notebook 4 completed successfully.")
    print("Required output created:")
    print("ENARES_2024_STAGE2_cloud_storage_report.md")

In [ ]:
# ============================================================
# 17. Create/update GitHub documentation for Stage 2
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"
DOCS_DIR = Path(f"{ROOT_DRIVE}/docs")

DOCS_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ID = input("Enter your Google Cloud PROJECT_ID: ").strip()
now = datetime.now(timezone.utc).isoformat()

docs_to_create = {
    "ENARES_2024_STAGE2_cloud_storage_workplan.md": f"""# ENARES 2024 CRS04 - Stage 2 Cloud Storage Workplan

Fecha: {now}

## Objetivo

Cargar CRS04 raw a BigQuery y preservar metadata SPSS/PDF sin transformaciones analiticas.

## Alcance

Stage 2 incluye:
- Crear/verificar datasets `raw`, `cleaned` y `analytical`.
- Cargar los cuatro archivos `.sav` CRS04 como tablas raw.
- Validar conteos de filas.
- Validar conteos de columnas/schema.
- Preservar metadata SPSS como tablas separadas.
- Registrar archivos fuente y PDFs oficiales cuando esten disponibles.
- Generar reporte de cierre auditable.

Stage 2 no incluye:
- Merge.
- Recodificacion.
- Variables derivadas.
- Indicadores.
- Modelos.
- Interpretacion estadistica.
""",

    "ENARES_2024_STAGE2_work_log.md": f"""# ENARES 2024 CRS04 - Stage 2 Work Log

Fecha: {now}

## Trabajo realizado

- Notebook 1: se crearon/verificaron datasets BigQuery.
- Notebook 2: se cargaron cuatro tablas raw CRS04.
- Notebook 3: se preservo metadata SPSS en tablas BigQuery.
- Notebook 4: se genero reporte de cierre Stage 2.

## Evidencia principal

Los outputs estan en `05Resultados/logs`.

## Resultado

Stage 2 aprobado segun `ENARES_2024_STAGE2_cloud_storage_report.md`.

## Tablas raw verificadas

- `raw_crs04_cap100`: 18,807 filas.
- `raw_crs04_cap200`: 18,807 filas.
- `raw_crs04_cap248`: 18,807 filas.
- `raw_crs04_cap300`: 18,807 filas.

## Incidencias

- Los archivos `.sav` estaban dentro de subcarpetas; se ajusto el notebook para buscarlos de forma recursiva.
""",

    "ENARES_2024_STAGE2_decision_log.md": f"""# ENARES 2024 CRS04 - Stage 2 Decision Log

Fecha: {now}

## Decision 1

Stage 2 solo carga, preserva y valida la capa raw.

## Motivo

La separacion `raw` / `cleaned` / `analytical` evita mezclar almacenamiento, limpieza estructural y analisis.

## Decision 2

No se ejecuta merge en Stage 2.

## Motivo

La llave de merge debe investigarse y demostrarse con evidencia en Stage 3.

## Hipotesis pendientes para Stage 3

- `COLEGIAL_ID`
- `ID`
- `ID + COLEGIAL_ID`
- Otra llave identificadora documentada en metadata

## Nota tecnica

Si `ID`, `COLEGIAL_ID` u otra llave llega como `FLOAT64`, se debe verificar que no existan decimales reales antes de convertir a `INT64`.

SPSS requiere `SORT CASES` antes de `MATCH FILES`; BigQuery no requiere ordenar antes de un `JOIN` porque SQL une por valores de llave, no por posicion.
""",

    "ENARES_2024_STAGE2_bigquery_registry.md": f"""# ENARES 2024 CRS04 - Stage 2 BigQuery Registry

Fecha: {now}

Google Cloud project_id: `{PROJECT_ID}`

## Datasets

- `{PROJECT_ID}.enares2024_crs04_raw`
- `{PROJECT_ID}.enares2024_crs04_cleaned`
- `{PROJECT_ID}.enares2024_crs04_analytical`

## Tablas raw

- `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100`
- `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap200`
- `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap248`
- `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap300`

## Tablas metadata

- `{PROJECT_ID}.enares2024_crs04_raw.metadata_crs04_variables`
- `{PROJECT_ID}.enares2024_crs04_raw.metadata_crs04_value_labels`
- `{PROJECT_ID}.enares2024_crs04_raw.metadata_crs04_missing_codes`
- `{PROJECT_ID}.enares2024_crs04_raw.metadata_crs04_source_files`

## Outputs de evidencia

- `ENARES_2024_STAGE2_bigquery_dataset_registry.csv`
- `ENARES_2024_STAGE2_crs04_bigquery_table_mapping.csv`
- `ENARES_2024_STAGE2_source_file_check.csv`
- `ENARES_2024_STAGE2_raw_table_inventory.csv`
- `ENARES_2024_STAGE2_rowcount_validation.csv`
- `ENARES_2024_STAGE2_schema_validation.csv`
- `ENARES_2024_STAGE2_metadata_inventory.csv`
- `ENARES_2024_STAGE2_cloud_storage_report.md`
"""
}

created = []

for filename, content in docs_to_create.items():
    path = DOCS_DIR / filename
    path.write_text(content, encoding="utf-8")
    created.append(str(path))

for path in created:
    print("Created/updated:", path)

In [ ]:
list(DOCS_DIR.glob("ENARES_2024_STAGE2*.md"))